# mPES - Colab Launcher desde Google Drive

Este notebook copia `h1/` y `utils/` desde Google Drive al disco local de
Colab y ejecuta una optimizacion alli. Los resultados se conservan en Drive.

Estructura requerida en Drive:

```text
MyDrive/mPES/
├── h1/
└── utils/
    └── config/requirements.txt
```

Para `ens_sprb` o `ens_accq`, la copia completa de `h1` debe contener los tres
modelos en sus rutas canonicas: `h1/ml/pes_dqn/inputs/dqn_model.keras`,
`h1/ml/pes_rdqn/inputs/rdqn_model.keras` y
`h1/ml/pes_trf/inputs/trf_model.keras`.

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab.

In [ ]:
"""Configure a Colab optimisation from h1 and utils stored on Drive."""
import os
import shutil
import subprocess

DRIVE_ROOT  = '/content/drive/MyDrive/mPES'
DRIVE_H1    = os.path.join(DRIVE_ROOT, 'h1')
DRIVE_UTILS = os.path.join(DRIVE_ROOT, 'utils')
WORKSPACE   = '/content/mPES'
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, 'runs')
PKG         = 'ens_sprb'  # ql | dql | dqn | rdqn | ac | tr | ens_sprb | ens_accq
N_TRIALS    = 50
RESUME_DATE = ''  # YYYY-MM-DD to resume, or '' for a new run
USE_GPU     = 0

valid_packages = ('ql', 'dql', 'dqn', 'rdqn', 'ac', 'tr', 'ens_sprb', 'ens_accq')
if PKG not in valid_packages:
    raise ValueError(f'Unsupported PKG: {PKG!r}')
if not os.path.isdir(DRIVE_H1):
    raise FileNotFoundError(f'Upload h1 to: {DRIVE_H1}')
if not os.path.isfile(os.path.join(DRIVE_UTILS, 'config', 'requirements.txt')):
    raise FileNotFoundError(f'Upload utils to: {DRIVE_UTILS}')

os.environ.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'DRIVE_H1': DRIVE_H1,
    'DRIVE_UTILS': DRIVE_UTILS,
    'H1_DIR': os.path.join(WORKSPACE, 'h1'),
    'REPO_DIR': WORKSPACE,
    'REQ_FILE': os.path.join(WORKSPACE, 'utils', 'config', 'requirements.txt'),
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
print(f'[CELL 1] PKG={PKG!r} N_TRIALS={N_TRIALS} DRIVE_H1={DRIVE_H1!r}')

In [ ]:
# ============================================================================
# Cell 2 - MOUNT GOOGLE DRIVE
# ============================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)


In [ ]:
# Copy h1 and utils to local Colab storage for faster file access.
local_h1 = os.path.join(WORKSPACE, 'h1')
local_utils = os.path.join(WORKSPACE, 'utils')
for local_path in (local_h1, local_utils):
    if os.path.isdir(local_path):
        shutil.rmtree(local_path)
os.makedirs(WORKSPACE, exist_ok=True)
shutil.copytree(DRIVE_H1, local_h1)
shutil.copytree(DRIVE_UTILS, local_utils)

subprocess.run(
    ['bash', os.path.join(local_h1, 'general', 'colab', 'setup_colab.sh')],
    check=True,
    env=os.environ.copy(),
)
print(f'[CELL 3] Local copies: {local_h1} and {local_utils}')

In [ ]:
_script = '''
set -uo pipefail
cd "$WORKSPACE_DIR"
source /content/mpes_env.sh
bash "$H1_DIR/general/colab/run_colab.sh" "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''
_proc = subprocess.run(
    _script, shell=True, executable='/bin/bash', check=False, env=os.environ.copy(),
)
if _proc.returncode != 0:
    raise RuntimeError(f'run_colab.sh exited with code {_proc.returncode}')